# 2.5D CoAtNet: one gold holdout, 750 MedGemma labels, single-model submission

Attach the RSNA competition data, the `coatnet_0_rw_224.sw_in1k` weights, and a private Kaggle dataset with `rsna_medgemma_modal27b_soft1000.csv`. The first split of the earlier five-fold setup is used as a single holdout: 46 gold training studies and 12 gold validation studies. All 750 report-labeled studies are training candidates; each surviving study is seen once across five epochs. The best checkpoint by gold validation BCE is used for `/kaggle/working/submission.csv`. This is one model and one holdout; its validation AUC is not directly comparable to the earlier 58-study OOF AUC.

The Modal labeling export retains the name `rsna_medgemma_modal27b_soft1000.csv` when stopped early. Attach the export containing exactly 750 valid rows; do not rename it. This notebook checks MedGemma 27B provenance and the original four-status soft-target schema.


In [1]:
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
import time, random, gc
import numpy as np
import pandas as pd
import torch, cv2, pydicom, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark=True
LABEL_COLS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
IMAGE_SIZE=224; CENTERS_PER_PLANE=8; IMAGE_MICROBATCH=3
NUM_DICOM_THREADS=8; NUM_WORKERS=2; N_FOLDS=5; EPOCHS=5
PSEUDO_LOSS_WEIGHT=.2; PSEUDO_PER_EPOCH=150
LR_BACKBONE=1e-5; LR_HEAD=2e-4
DATA_CANDIDATES=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
DATA_ROOT=next((p for p in DATA_CANDIDATES if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('Attach RSNA Knee Abnormality Detection competition data.')
TRAIN_SERIES_ROOT=DATA_ROOT/'train_series'; TEST_SERIES_ROOT=DATA_ROOT/'test_series'
OUT=Path('/kaggle/working'); CACHE=OUT/'coatnet_2p5d_medgemma500_cache'; CACHE.mkdir(exist_ok=True)
EXPECTED_WEAK_ROWS=750
WEAK_LABEL_CSV= Path("/kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1000.csv")  # Set the exact attached CSV path if shallow discovery cannot find it.
COATNET_WEIGHTS= Path('/kaggle/input/models/amritgopinath/coatnet-fine-tuned/pytorch/default/1/coatnet_kneemri_backbone.pt')
print('device',DEVICE,'timm',timm.__version__)


device cuda timm 1.0.26


In [2]:
def find_coatnet():
    for top in Path('/kaggle/input').iterdir():
        if top.is_dir() and 'coatnet' in top.name.lower():
            for name in ('model.safetensors','pytorch_model.bin','coatnet_0_rw_224.pth'):
                matches=list(top.rglob(name))
                if matches: return matches[0]
    return None
COATNET_WEIGHTS=Path(COATNET_WEIGHTS) if COATNET_WEIGHTS else find_coatnet()
if COATNET_WEIGHTS is None: raise FileNotFoundError('Attach official timm coatnet_0_rw_224.sw_in1k weights, then set COATNET_WEIGHTS.')
print('ImageNet weights',COATNET_WEIGHTS)


ImageNet weights /kaggle/input/models/amritgopinath/coatnet-fine-tuned/pytorch/default/1/coatnet_kneemri_backbone.pt


In [3]:
train=pd.read_csv(DATA_ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(DATA_ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test_series=pd.read_csv(DATA_ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample_submission=pd.read_csv(DATA_ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert list(sample_submission.columns)==['StudyInstanceUID']+LABEL_COLS
gold=train[train[LABEL_COLS].notna().all(axis=1)].copy().reset_index(drop=True)
gold[LABEL_COLS]=gold[LABEL_COLS].apply(pd.to_numeric)
assert len(gold)==58, f'Expected 58 gold studies; got {len(gold)}'
assert gold.StudyInstanceUID.is_unique
print('gold studies',len(gold),'test studies',len(sample_submission))

# Load the actual Modal export. No recursive search through RSNA DICOM folders.
import hashlib
csv_name='rsna_medgemma_modal27b_soft1000.csv'
if WEAK_LABEL_CSV is not None:
    csv_paths=[Path(WEAK_LABEL_CSV)]
else:
    input_root=Path('/kaggle/input')
    candidates=[OUT/csv_name]
    # Only fixed-depth dataset layouts; never use rglob here.
    candidates.extend(input_root.glob('*/'+csv_name))
    candidates.extend(input_root.glob('datasets/*/*/'+csv_name))
    csv_paths=list(dict.fromkeys(p for p in candidates if p.is_file()))
if len(csv_paths)!=1 or not csv_paths[0].is_file():
    raise FileNotFoundError(
        f'Attach exactly one {csv_name}, or set WEAK_LABEL_CSV to its exact path; '
        f'found {len(csv_paths)}: {csv_paths[:3]}'
    )
print('Reading weak labels from',csv_paths[0])
pseudo=pd.read_csv(csv_paths[0],dtype={'StudyInstanceUID':str,'report_sha256':str})
assert len(pseudo)==EXPECTED_WEAK_ROWS and pseudo.StudyInstanceUID.is_unique, f'Expected 750 unique rows; got {len(pseudo)}'
required={'StudyInstanceUID','report_sha256','model','prompt_version',*LABEL_COLS,
          *('weight__'+x for x in LABEL_COLS),*('status__'+x for x in LABEL_COLS)}
assert required.issubset(pseudo.columns), f'Missing CSV columns: {required-set(pseudo.columns)}'
assert 'Report' in train.columns and train.StudyInstanceUID.is_unique
assert set(pseudo.StudyInstanceUID).isdisjoint(set(gold.StudyInstanceUID)), 'Gold study in weak labels'
assert set(pseudo.StudyInstanceUID).issubset(set(train.StudyInstanceUID)), 'Unexpected study ID'
assert pseudo.model.eq('google/medgemma-27b-text-it').all()
assert pseudo.prompt_version.eq('medgemma-knee-v1-modal-text-v1').all()
report_lookup=train.set_index('StudyInstanceUID')['Report']
for row in pseudo[['StudyInstanceUID','report_sha256']].itertuples(index=False):
    digest=hashlib.sha256(str(report_lookup.loc[row.StudyInstanceUID]).encode('utf-8')).hexdigest()
    if row.report_sha256!=digest:
        raise ValueError(f'Report hash mismatch for study {row.StudyInstanceUID}')
soft_map={'positive':(.95,1.),'negative':(.05,1.),'uncertain':(.5,.2),'not_mentioned':(.5,0.)}
for col in LABEL_COLS:
    values=pd.to_numeric(pseudo[col],errors='raise')
    weights=pd.to_numeric(pseudo['weight__'+col],errors='raise')
    expected=pseudo['status__'+col].map(soft_map)
    if expected.isna().any(): raise ValueError(f'Invalid status for {col}')
    assert np.allclose(values.to_numpy(),expected.map(lambda v:v[0]).to_numpy())
    assert np.allclose(weights.to_numpy(),expected.map(lambda v:v[1]).to_numpy())
    pseudo[col]=values.astype(float)
    pseudo['weight__'+col]=weights.astype(float)
pseudo=pseudo.sample(frac=1,random_state=SEED).reset_index(drop=True)
print('report-only pseudo rows',len(pseudo),'usable targets',int(pseudo[['weight__'+x for x in LABEL_COLS]].gt(0).sum().sum()))


gold studies 58 test studies 3
Reading weak labels from /kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1000.csv
report-only pseudo rows 750 usable targets 6310


In [4]:
from safetensors.torch import load_file as load_safetensors

def make_backbone(load_pretrained=True):
    net=timm.create_model('coatnet_0_rw_224.sw_in1k',pretrained=False)
    net.reset_classifier(0)  # Match a feature-only or full ImageNet checkpoint.
    if load_pretrained and COATNET_WEIGHTS.suffix=='.safetensors': state=load_safetensors(str(COATNET_WEIGHTS),device='cpu')
    elif load_pretrained:
        state=torch.load(COATNET_WEIGHTS,map_location='cpu',weights_only=True)
        if isinstance(state,dict): state=state.get('state_dict',state.get('model',state))
        state={k.removeprefix('module.'):v for k,v in state.items()}
    if load_pretrained:
        # Classifier keys are optional and unused for the 12-label task.
        # Keep strict=True for every remaining backbone weight and shape.
        state={k.removeprefix('module.'):v for k,v in state.items()}
        for key in ('head.fc.weight','head.fc.bias'):
            state.pop(key,None)
        net.load_state_dict(state,strict=True)
    for p in net.parameters(): p.requires_grad_(False)
    if not hasattr(net,'stages'): raise RuntimeError('Unexpected timm CoAtNet layout; inspect net and choose its final stage.')
    for p in net.stages[-1].parameters(): p.requires_grad_(True)
    return net

class StudyCoAtNet(nn.Module):
    def __init__(self,load_pretrained=True):
        super().__init__()
        self.backbone=make_backbone(load_pretrained)
        dim=self.backbone.num_features
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(.15),nn.Linear(dim,len(LABEL_COLS)))
        self.plane_weight=nn.Parameter(torch.zeros(4,len(LABEL_COLS)))
        self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))

    def forward(self,images,planes):
        slice_logits=[]
        for chunk in images.split(IMAGE_MICROBATCH):
            x=(chunk.to(dtype=self.mean.dtype)/255-self.mean)/self.std
            ctx=torch.autocast('cuda',dtype=torch.float16) if self.training and images.is_cuda else nullcontext()
            with ctx: slice_logits.append(self.head(self.backbone(x)))
        scores=torch.cat(slice_logits,0)
        pooled=[]; available=[]
        for plane in range(4):
            part=scores[planes==plane]
            available.append(len(part)>0)
            pooled.append(part.topk(min(2,len(part)),dim=0).values.mean(0) if len(part) else torch.zeros_like(scores[0]))
        mask=torch.tensor(available,device=images.device)
        weights=self.plane_weight.masked_fill(~mask[:,None],-1e4).softmax(0)
        return (weights*torch.stack(pooled)).sum(0)

# Check checkpoint compatibility before the long DICOM caching stage.
_probe_backbone=make_backbone(load_pretrained=True)
del _probe_backbone
print("CoAtNet backbone checkpoint loaded successfully")


CoAtNet backbone checkpoint loaded successfully


In [5]:
def plane_of(row):
    p=str(row.get('Anatomical_Plane','')).strip().upper()
    if p.startswith('SAG'): return 'SAGITTAL'
    if p.startswith('COR'): return 'CORONAL'
    if p.startswith('AX'): return 'AXIAL'
    return 'UNKNOWN'

def select_series(df, ids):
    df=df.copy()
    df['StudyInstanceUID']=df.StudyInstanceUID.astype(str)
    df['SeriesInstanceUID']=df.SeriesInstanceUID.astype(str)
    df=df[df.StudyInstanceUID.isin(set(map(str,ids)))].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    df['plane_norm']=df.apply(plane_of,axis=1)
    for col in ('Fluid_Sensitive','Fat_Suppression'):
        if col not in df: df[col]='UNKNOWN'
        df[col]=df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    rows=[]
    for sid,group in df.groupby('StudyInstanceUID',sort=True):
        group=group.sort_values('SeriesInstanceUID')
        picks=[]
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            if len(candidates): picks.append(candidates.iloc[0])
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            used=[p for p in picks if p.plane_norm==plane]
            if len(candidates)<2 or not used: continue
            first=used[0]
            rest=candidates[candidates.SeriesInstanceUID.ne(first.SeriesInstanceUID)]
            diverse=rest[(rest.Fluid_Sensitive.ne(first.Fluid_Sensitive)) | (rest.Fat_Suppression.ne(first.Fat_Suppression))]
            if len(diverse): picks.append(diverse.iloc[0])
            elif len(rest): picks.append(rest.iloc[0])
        if not picks and len(group): picks.append(group.iloc[0])
        rows.extend([{'StudyInstanceUID':sid,'SeriesInstanceUID':str(r.SeriesInstanceUID),'plane_norm':r.plane_norm} for r in picks])
    return pd.DataFrame(rows,columns=['StudyInstanceUID','SeriesInstanceUID','plane_norm'])

train_picks=select_series(train_series,pd.concat([gold.StudyInstanceUID,pseudo.StudyInstanceUID],ignore_index=True))
test_picks=select_series(test_series,sample_submission.StudyInstanceUID)
train_picks.to_csv(OUT/'coatnet_morecoverage_gold_series_selection.csv',index=False)
assert set(gold.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Gold study has no metadata series'
assert set(pseudo.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Pseudo study has no metadata series'
print('gold series',len(train_picks),'test series',len(test_picks))


gold series 4019 test series 14


In [6]:
def _read_one(path):
    try:
        d=pydicom.dcmread(str(path),force=True)
        x=np.squeeze(d.pixel_array)
        if x.ndim!=2: return None
        x=x.astype(np.float32)*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
        if str(getattr(d,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        o=np.asarray(getattr(d,'ImageOrientationPatient',[]),float)
        pos=np.asarray(getattr(d,'ImagePositionPatient',[]),float)
        g=np.nan
        if len(o)==6 and len(pos)==3:
            n=np.cross(o[:3],o[3:]); norm=np.linalg.norm(n)
            if norm>0: g=float(pos @ (n/norm))
        try: inst=float(d.InstanceNumber)
        except Exception: inst=np.nan
        return (path.name,g,inst,x)
    except Exception: return None

def resize_pad(x):
    h,w=x.shape; scale=IMAGE_SIZE/max(h,w)
    nh,nw=max(1,round(h*scale)),max(1,round(w*scale))
    z=cv2.resize(x,(nw,nh),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_LINEAR)
    out=np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.uint8)
    top=(IMAGE_SIZE-nh)//2; left=(IMAGE_SIZE-nw)//2
    out[top:top+nh,left:left+nw]=np.rint(z*255).astype(np.uint8)
    return out

def load_plane(study,series,root):
    directory=root/str(study)/str(series)
    paths=sorted(p for p in directory.iterdir() if p.is_file() and not p.name.startswith('.')) if directory.is_dir() else []
    if not paths: return None
    with ThreadPoolExecutor(max_workers=NUM_DICOM_THREADS) as pool: rows=[r for r in pool.map(_read_one,paths) if r is not None]
    if not rows: return None
    key=1 if all(np.isfinite(r[1]) for r in rows) else 2 if all(np.isfinite(r[2]) for r in rows) else 0
    rows.sort(key=lambda r:(r[key],r[0]) if key else (r[0],))
    samples=np.concatenate([r[3][::8,::8].ravel() for r in rows])
    samples=samples[np.isfinite(samples)]
    if not len(samples): return None
    lo,hi=np.percentile(samples,[1,99])
    n=len(rows); centers=np.rint(np.linspace(.05*(n-1),.95*(n-1),CENTERS_PER_PLANE)).astype(int)
    needed=sorted(set(np.clip(np.r_[centers-1,centers,centers+1],0,n-1)))
    maps={i:resize_pad(np.clip(np.nan_to_num(rows[i][3],nan=lo,posinf=hi,neginf=lo),lo,hi).astype(np.float32).clip(lo,hi).astype(np.float32)/(hi-lo) if hi>lo else np.zeros_like(rows[i][3],np.float32)) for i in needed}
    return np.stack([np.stack([maps[max(0,i-1)],maps[i],maps[min(n-1,i+1)]],0) for i in centers],0)

def cache_path(study,root):
    return CACHE/('train_' if root==TRAIN_SERIES_ROOT else 'test_')/f'{study}.npz'

def cache_study(study,picks,root):
    path=cache_path(study,root)
    if path.exists(): return path
    blocks=[]; planes=[]
    group=picks[picks.StudyInstanceUID.eq(str(study))]
    for row in group.itertuples(index=False):
        try: block=load_plane(study,row.SeriesInstanceUID,root)
        except Exception: block=None
        if block is None: continue
        blocks.append(block); planes.extend([('SAGITTAL','CORONAL','AXIAL').index(row.plane_norm) if row.plane_norm in ('SAGITTAL','CORONAL','AXIAL') else 3]*len(block))
    if not blocks: return None
    path.parent.mkdir(parents=True,exist_ok=True)
    np.savez(path,images=np.concatenate(blocks,0),planes=np.asarray(planes,np.int8))
    return path

def warm_cache(ids,picks,root):
    t=time.time(); failures=[]
    for i,sid in enumerate(map(str,ids),1):
        if cache_study(sid,picks,root) is None: failures.append(sid)
        if i%100==0 or i==len(ids): print(f'cached {i}/{len(ids)} in {(time.time()-t)/60:.1f} min; failed={len(failures)}')
    return failures


gold_failures=warm_cache(gold.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
assert not gold_failures, f'Gold studies failed DICOM decode: {gold_failures}'
pseudo_failures=warm_cache(pseudo.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
if pseudo_failures: print('Dropped undecodable pseudo studies:',len(pseudo_failures))
pseudo=pseudo.loc[~pseudo.StudyInstanceUID.isin(pseudo_failures)].reset_index(drop=True)
assert len(pseudo)>0, 'No pseudo studies decoded.'


cached 58/58 in 1.4 min; failed=0
cached 100/750 in 2.4 min; failed=0
cached 200/750 in 4.6 min; failed=0
cached 300/750 in 6.9 min; failed=0
cached 400/750 in 8.8 min; failed=0
cached 500/750 in 11.2 min; failed=0
cached 600/750 in 13.3 min; failed=0
cached 700/750 in 15.6 min; failed=0
cached 750/750 in 16.6 min; failed=0


In [7]:
class BagDS(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with np.load(cache_path(str(row.StudyInstanceUID),TRAIN_SERIES_ROOT)) as d:
            x=d['images'].copy(); plane=d['planes'].astype(np.int64)
        return torch.from_numpy(x),torch.from_numpy(plane),torch.tensor(row[LABEL_COLS].to_numpy(np.float32))

class PseudoBagDS(BagDS):
    def __getitem__(self,i):
        x,plane,y=super().__getitem__(i)
        row=self.frame.iloc[i]
        weight=torch.tensor(row[['weight__'+c for c in LABEL_COLS]].to_numpy(np.float32))
        return x,plane,y,weight

def train_fold(model,frame,validation,fold,checkpoint):
    gold_loader=DataLoader(BagDS(frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                      collate_fn=lambda b:b[0])
    optimizer=torch.optim.AdamW([{'params':model.backbone.stages[-1].parameters(),'lr':LR_BACKBONE},
                                 {'params':list(model.head.parameters())+[model.plane_weight],'lr':LR_HEAD}],weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
    best_bce=np.inf; best_epoch=None; history=[]
    # The same pseudo order is shared by folds, but the model and gold split differ.
    order=np.random.default_rng(SEED).permutation(len(pseudo))
    weak_chunks=np.array_split(order,EPOCHS)
    for epoch in range(1,EPOCHS+1):
        model.train(); model.backbone.eval(); model.backbone.stages[-1].train()
        losses=[]; start=time.time()
        weak_frame=pseudo.iloc[weak_chunks[epoch-1]]
        assert len(weak_frame)>0
        weak_loader=DataLoader(PseudoBagDS(weak_frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                          collate_fn=lambda b:b[0])
        gold_iter=iter(gold_loader); weak_iter=iter(weak_loader)
        schedule=np.array([0]*len(frame)+[1]*len(weak_frame))
        np.random.default_rng(SEED+fold*100+epoch).shuffle(schedule)
        for mode in schedule:
            if mode==0:
                x,plane,y=next(gold_iter); weights=None
            else:
                x,plane,y,weights=next(weak_iter)
            x=x.to(DEVICE,non_blocking=True); plane=plane.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,plane).float()
            if mode==0:
                loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
            else:
                weights=weights.to(DEVICE)
                raw=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
                # Preserve uncertain/not-mentioned masking and don't inflate a study with few asserted findings.
                loss=PSEUDO_LOSS_WEIGHT*(raw*weights).sum()/weights.numel()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
        pred,failed=predict_ids(model,validation.StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        if failed: raise RuntimeError(f'Fold {fold} validation decode failures: {failed}')
        aligned=validation[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        y=validation[LABEL_COLS].to_numpy(np.float32)
        probs=aligned[LABEL_COLS].to_numpy(np.float32).clip(1e-6,1-1e-6)
        val_bce=nn.functional.binary_cross_entropy(torch.from_numpy(probs),torch.from_numpy(y)).item()
        aucs=[roc_auc_score(y[:,j],probs[:,j]) for j in range(len(LABEL_COLS)) if np.unique(y[:,j]).size==2]
        val_auc=float(np.mean(aucs)) if aucs else float('nan')
        if val_bce<best_bce-1e-6:
            best_bce=val_bce; best_epoch=epoch
            torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},checkpoint)
        history.append({'fold':fold,'epoch':epoch,'train_weighted_loss':float(np.mean(losses)),
                        'val_bce':val_bce,'val_macro_auc':val_auc,'n_auc_labels':len(aucs),
                        'best_so_far':best_epoch,'minutes':(time.time()-start)/60})
        print(f'fold {fold} epoch {epoch}/{EPOCHS}: weighted loss {np.mean(losses):.4f}, '
              f'val BCE {val_bce:.4f}, val AUC {val_auc:.4f} ({len(aucs)} labels), '
              f'best epoch {best_epoch}, time {(time.time()-start)/60:.1f}m')
    model.load_state_dict(torch.load(checkpoint,map_location='cpu',weights_only=True),strict=True)
    return best_epoch,best_bce,history

@torch.inference_mode()
def predict_ids(model,ids,picks,root):
    model.eval(); rows=[]; failed=[]
    for sid in map(str,ids):
        path=cache_path(sid,root)
        if not path.exists(): cache_study(sid,picks,root)
        if not path.exists(): failed.append(sid); continue
        with np.load(path) as d:
            x=torch.from_numpy(d['images'].copy()).to(DEVICE); plane=torch.from_numpy(d['planes'].astype(np.int64)).to(DEVICE)
        score=torch.sigmoid(model(x,plane).float()).cpu().numpy()
        rows.append([sid,*score.tolist()])
    return pd.DataFrame(rows,columns=['StudyInstanceUID']+LABEL_COLS),failed

# Match the first held-out gold split from the five-fold notebook, but train only one model.
tr,va=next(KFold(n_splits=5,shuffle=True,random_state=SEED).split(gold))
fold=1
print(f'SINGLE FOLD: {len(tr)} gold training, {len(pseudo)} decodable weak training, {len(va)} gold validation')
assert len(tr)==46 and len(va)==12
split=gold[['StudyInstanceUID']].copy()
split['split']='gold_train'; split.loc[va,'split']='gold_validation'
split.to_csv(OUT/'coatnet_medgemma750_singlefold_split.csv',index=False)
model=StudyCoAtNet(load_pretrained=True).to(DEVICE)
checkpoint=OUT/'coatnet_medgemma750_singlefold_best.pt'
best_epoch,best_bce,epoch_history=train_fold(model,gold.iloc[tr],gold.iloc[va],fold,checkpoint)
pred,failed=predict_ids(model,gold.iloc[va].StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
assert not failed
aligned=gold.iloc[va][['StudyInstanceUID']+LABEL_COLS].merge(pred,on='StudyInstanceUID',validate='one_to_one',suffixes=('__true','__pred'))
aligned.to_csv(OUT/'coatnet_medgemma750_singlefold_validation_predictions.csv',index=False)
pd.DataFrame(epoch_history).to_csv(OUT/'coatnet_medgemma750_singlefold_epoch_history.csv',index=False)
metrics=[]
for label in LABEL_COLS:
    y=aligned[label+'__true'].to_numpy(float)
    p=aligned[label+'__pred'].to_numpy(float)
    metrics.append({'label':label,'validation_positives':int(y.sum()),
                    'auc':roc_auc_score(y,p) if np.unique(y).size==2 else np.nan})
metrics=pd.DataFrame(metrics)
metrics.to_csv(OUT/'coatnet_medgemma750_singlefold_validation_auc.csv',index=False)
print('Best checkpoint',checkpoint,'epoch',best_epoch,'validation BCE',best_bce)
print('Validation macro AUC over eligible labels',metrics.auc.mean())
display(metrics)
del model; gc.collect()
if DEVICE.type=='cuda': torch.cuda.empty_cache()


SINGLE FOLD: 46 gold training, 750 decodable weak training, 12 gold validation


/tmp/ipykernel_22/3469277163.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 1 epoch 1/5: weighted loss 0.2240, val BCE 0.6006, val AUC 0.4711 (10 labels), best epoch 1, time 1.1m
fold 1 epoch 2/5: weighted loss 0.2076, val BCE 0.5969, val AUC 0.5157 (10 labels), best epoch 2, time 0.9m
fold 1 epoch 3/5: weighted loss 0.1946, val BCE 0.5717, val AUC 0.5275 (10 labels), best epoch 3, time 0.9m
fold 1 epoch 4/5: weighted loss 0.1845, val BCE 0.5951, val AUC 0.5455 (10 labels), best epoch 3, time 0.9m
fold 1 epoch 5/5: weighted loss 0.1699, val BCE 0.5777, val AUC 0.5480 (10 labels), best epoch 3, time 0.9m
Best checkpoint /kaggle/working/coatnet_medgemma750_singlefold_best.pt epoch 3 validation BCE 0.5716872811317444
Validation macro AUC over eligible labels 0.5274570105820106


,label,validation_positives,auc
0,ACL,3,0.777778
1,MCL,3,0.185185
2,Medial Meniscus,5,0.742857
3,Lateral Meniscus,4,0.625000
4,Medial OA,4,0.437500
5,Lateral OA,0,NaN
6,PF OA,4,0.218750
7,Effusion,8,0.812500
8,Synovitis,4,0.625000
9,Baker's,2,0.350000


In [8]:
# One checkpoint supplies the entire test submission.
test_ids=sample_submission.StudyInstanceUID.astype(str).tolist()
test_failures=warm_cache(test_ids,test_picks,TEST_SERIES_ROOT)
print('test decode failures',len(test_failures))
model=StudyCoAtNet(load_pretrained=False)
state=torch.load(checkpoint,map_location='cpu',weights_only=True)
model.load_state_dict(state,strict=True)
model=model.to(DEVICE)
if DEVICE.type=='cuda': model=model.half()
pred,failed=predict_ids(model,test_ids,test_picks,TEST_SERIES_ROOT)
aligned_test=sample_submission[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',how='left',validate='one_to_one')
fallback=gold[LABEL_COLS].mean().to_numpy(float)
for j,label in enumerate(LABEL_COLS):
    aligned_test[label]=aligned_test[label].fillna(float(fallback[j]))
submission=sample_submission[['StudyInstanceUID']].copy()
submission[LABEL_COLS]=aligned_test[LABEL_COLS].to_numpy(np.float32)
assert list(submission.columns)==list(sample_submission.columns)
assert submission.StudyInstanceUID.is_unique and len(submission)==len(sample_submission)
assert np.isfinite(submission[LABEL_COLS].to_numpy()).all()
assert submission[LABEL_COLS].to_numpy().min()>=0 and submission[LABEL_COLS].to_numpy().max()<=1
submission.to_csv(OUT/'submission.csv',index=False)
print('Single checkpoint predicted',len(pred),'test studies; fallback used for',len(failed))
print('wrote',OUT/'submission.csv','shape',submission.shape)
display(submission.head())


cached 3/3 in 0.1 min; failed=0
test decode failures 0
Single checkpoint predicted 3 test studies; fallback used for 0
wrote /kaggle/working/submission.csv shape (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.306112,0.173009,0.450450,0.305387,0.419767,0.200401,0.415847,0.397588,0.666822,0.380874,0.366930,0.403567
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.283781,0.173148,0.335133,0.194062,0.143427,0.157655,0.177098,0.449302,0.336113,0.129623,0.242923,0.448607
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.237222,0.195134,0.303113,0.224905,0.342901,0.189671,0.319121,0.377082,0.495503,0.241311,0.196363,0.428233
